# Week 4 Day 2: Guided Practice - Production Pipeline with Wine Quality

**Dataset:** Wine Quality UCI  
**Goal:** Build complete production ML pipeline using Week 4's four pillars  
**Scaffolding Level:** 70% provided (guided practice)  
**Time Estimate:** 20 minutes (in-class)

---

## How to Use This Notebook

🎯 **Your Focus:** Applying production methodologies (not coding from scratch)

✅ **Code is mostly provided** - Your job is to:
- Fill in Pipeline components
- Complete GridSearchCV param_grid
- Choose appropriate CV strategy
- Interpret production-ready results

👨‍🏫 **Instructor Support:**
- Watch for **"PAUSE HERE"** - instructor will explain
- Look for **"YOUR TASK"** - you'll work independently
- **"DISCUSS"** - talk with neighbor

---

## Learning Objectives

By completing this exercise, you will:
1. Build end-to-end Pipeline preventing data leakage
2. Use GridSearchCV for systematic hyperparameter tuning
3. Apply cross-validation for reliable performance estimates
4. Integrate regularization for overfitting prevention

---

## The Challenge

You are a data scientist at a winery. Your task is to build a **production-ready** model predicting wine quality (good vs bad) from physicochemical properties.

**Critical Requirements:**
- NO data leakage (use Pipeline)
- Systematic tuning (use GridSearchCV)
- Reliable estimates (use cross-validation)
- Prevent overfitting (use regularization)

**Key Question:** Can you build a production pipeline that meets industry standards?

---

## Part 1: Setup and Data Loading

**Code provided below** - Just run the cells and observe the output.

In [ ]:
# Import all necessary libraries (ALL PROVIDED)

import numpy as np
import pandas as pd

# sklearn - preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# sklearn - models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# sklearn - metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Model persistence
import joblib

%matplotlib inline

print("✅ All imports successful!")

In [ ]:
# Load Wine Quality dataset (ALL PROVIDED)

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'

df = pd.read_csv(url, sep=';')

print("✅ Data loaded!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

**PAUSE HERE** - Instructor will explain wine quality features (pH, alcohol, acidity, etc.)

In [ ]:
# Check data info (ALL PROVIDED)

print("Dataset Info:")
print(df.info())
print("\nTarget distribution:")
print(df['quality'].value_counts().sort_index())

In [ ]:
# Convert to binary classification (ALL PROVIDED)
# Quality >= 6 = good wine (1), Quality < 6 = bad wine (0)

df['quality_binary'] = (df['quality'] >= 6).astype(int)

print("Binary quality distribution:")
print(df['quality_binary'].value_counts())
print(f"\nClass balance: {df['quality_binary'].mean():.2%} good wines")

**DISCUSS with neighbor (1 minute):** Is this dataset balanced or imbalanced? Should we use stratified CV?

In [ ]:
# Prepare features and target (ALL PROVIDED)

X = df.drop(['quality', 'quality_binary'], axis=1)
y = df['quality_binary']

print(f"✅ Data prepared!")
print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")

---

## Part 2: Train/Test Split - YOUR TURN! 🎯

**YOUR TASK:** Complete the train_test_split with appropriate parameters.

**Remember Week 4 principles:**
- Split BEFORE any transformations (prevent leakage)
- Use stratify for imbalanced classification
- Standard test_size = 0.2 (20%)

In [ ]:
# TODO: Complete train_test_split parameters
# Hint: Should you use stratify? Why or why not?

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=____,      # Fill in: standard test size (0.2)
    random_state=42,
    stratify=____        # Fill in: stratify=y for imbalanced data, or None
)

print("✅ Data split complete!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTrain class balance: {y_train.mean():.2%}")
print(f"Test class balance: {y_test.mean():.2%}")

**PAUSE HERE** - Instructor will explain why we split BEFORE scaling (data leakage prevention).

---

## Part 3: Build Pipeline - YOUR TURN! 🎯

**YOUR TASK:** Complete the Pipeline with scaler and model.

**Pipeline Structure:**
```
Pipeline([
    ('scaler', StandardScaler()),           # Normalize features
    ('classifier', LogisticRegression())    # Model
])
```

**Why Pipeline?** Ensures scaler fits on each CV fold independently (prevents leakage).

In [ ]:
# TODO: Complete Pipeline components
# Hint: Pipeline takes a LIST of tuples: [('name', transformer), ('name', model)]

pipeline = Pipeline([
    ('____', ____),  # Fill in: ('scaler', StandardScaler())
    ('____', ____)   # Fill in: ('classifier', LogisticRegression(max_iter=1000))
])

print("✅ Pipeline created!")
print(pipeline)

**DISCUSS with neighbor (1 minute):** What would happen if we scaled BEFORE splitting? Why is Pipeline better?

---

## Part 4: Cross-Validation Baseline - YOUR TURN! 🎯

**YOUR TASK:** Use cross_val_score to get baseline performance.

**Remember:**
- Use training data only (X_train, y_train)
- Choose appropriate cv strategy (regular or stratified)
- Report mean ± std

In [ ]:
# TODO: Complete cross_val_score parameters
# Hint: Should cv be an integer (e.g., 5) or StratifiedKFold(5)?

cv_scores = cross_val_score(
    pipeline,
    ____,            # Fill in: X_train
    ____,            # Fill in: y_train
    cv=____,         # Fill in: StratifiedKFold(5) for imbalanced data
    scoring='accuracy'
)

print("✅ Cross-validation complete!")
print(f"\nBaseline CV Scores: {cv_scores}")
print(f"Mean: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

**PAUSE HERE** - Instructor will explain how to interpret mean ± std notation.

---

## Part 5: GridSearchCV for Hyperparameter Tuning - YOUR TURN! 🎯

**YOUR TASK:** Complete param_grid to tune LogisticRegression's C (regularization strength).

**Key Concepts:**
- C = inverse of regularization strength
- Small C (e.g., 0.01) = strong regularization (simpler model)
- Large C (e.g., 100) = weak regularization (complex model)
- Try range: [0.01, 0.1, 1, 10, 100]

**Pipeline parameter syntax:** `'classifier__C'` (double underscore)

In [ ]:
# TODO: Complete param_grid for tuning regularization strength C
# Hint: Use 'classifier__C' to access Pipeline's classifier hyperparameter
# Hint: Try values [0.01, 0.1, 1, 10, 100]

param_grid = {
    'classifier__C': ____  # Fill in: [0.01, 0.1, 1, 10, 100]
}

print("✅ param_grid defined!")
print(param_grid)

In [ ]:
# TODO: Complete GridSearchCV setup
# Hint: Use cv=StratifiedKFold(5) for imbalanced data

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=____,             # Fill in: StratifiedKFold(5)
    scoring='accuracy',
    n_jobs=-1,           # Use all CPU cores
    verbose=1
)

# Fit GridSearchCV (PROVIDED)
print("\n🔍 Starting GridSearchCV...")
grid_search.fit(X_train, y_train)

print("\n✅ GridSearchCV complete!")

**PAUSE HERE** - Instructor will explain what GridSearchCV just did (nested CV for tuning).

In [ ]:
# Extract best results (ALL PROVIDED)

print("GridSearchCV Results:")
print("=" * 60)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")
print("=" * 60)

# Show all results
results_df = pd.DataFrame(grid_search.cv_results_)
print("\nAll parameter combinations:")
print(results_df[['param_classifier__C', 'mean_test_score', 'std_test_score']].sort_values('mean_test_score', ascending=False))

**DISCUSS with neighbor (2 minutes):** 
1. Which C value performed best?
2. Does strong regularization (small C) or weak regularization (large C) work better here?
3. What does this tell you about model complexity needs?

---

## Part 6: Final Evaluation on Test Set (PROVIDED)

**Important:** Test set used ONCE at the end (simulates production deployment).

In [ ]:
# Evaluate best model on test set (ALL PROVIDED)

best_pipeline = grid_search.best_estimator_
y_pred = best_pipeline.predict(X_test)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]

# Calculate metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_pred_proba)

print("Final Test Set Performance:")
print("=" * 60)
print(f"Test Accuracy: {test_accuracy:.3f}")
print(f"Test AUC-ROC:  {test_auc:.3f}")
print("=" * 60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Bad Wine', 'Good Wine']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Bad', 'Good'], 
            yticklabels=['Bad', 'Good'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix - Test Set')
plt.show()

---

## Part 7: Production Deployment - Save Pipeline (PROVIDED)

**In production:** Save trained pipeline with joblib for deployment.

In [ ]:
# Save trained pipeline (ALL PROVIDED)

model_filename = 'wine_quality_pipeline.pkl'
joblib.dump(best_pipeline, model_filename)

print(f"✅ Pipeline saved to {model_filename}")
print(f"\nFile size: {os.path.getsize(model_filename) / 1024:.2f} KB")

In [ ]:
# Test loading saved pipeline (ALL PROVIDED)

loaded_pipeline = joblib.load(model_filename)

# Make a test prediction
test_sample = X_test.iloc[0:1]
prediction = loaded_pipeline.predict(test_sample)
probability = loaded_pipeline.predict_proba(test_sample)

print("✅ Pipeline loaded successfully!")
print(f"\nTest prediction:")
print(f"Features: {test_sample.values[0]}")
print(f"Predicted class: {'Good Wine' if prediction[0] == 1 else 'Bad Wine'}")
print(f"Probabilities: Bad={probability[0][0]:.3f}, Good={probability[0][1]:.3f}")

---

## Part 8: Data Leakage Check - YOUR TURN! 🎯

**YOUR TASK:** Review the WRONG code below and identify the data leakage bug.

In [ ]:
# WRONG CODE - Find the leakage bug!
"""
# Load data
X = df.drop(['quality', 'quality_binary'], axis=1)
y = df['quality_binary']

# Scale ALL data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # ← LEAKAGE HERE!

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)
"""

print("⚠️ WRONG CODE ABOVE - Do NOT run this!\n")
print("YOUR TASK: Identify the data leakage bug.\n")
print("Questions to answer:")
print("1. What line causes data leakage?")
print("2. Why is this a problem?")
print("3. How does Pipeline fix this?")

### YOUR ANSWERS:

**1. What line causes data leakage?**

Your answer: _______________________________________________________________

**2. Why is this a problem?**

Your answer:

_____________________________________________________________________

_____________________________________________________________________

**3. How does Pipeline fix this?**

Your answer:

_____________________________________________________________________

_____________________________________________________________________

**PAUSE HERE** - Instructor will lead class discussion on leakage prevention.

---

## 🎉 Congratulations!

You've completed the Week 4 Day 2 guided practice!

**What you accomplished:**
1. ✅ Built production Pipeline preventing data leakage
2. ✅ Used GridSearchCV for systematic hyperparameter tuning
3. ✅ Applied cross-validation for reliable performance estimates
4. ✅ Integrated regularization for overfitting prevention
5. ✅ Saved trained pipeline for deployment
6. ✅ Identified data leakage bugs

---

## Key Takeaways

📌 **Pipeline** - Prevents data leakage by applying transformations per fold

📌 **GridSearchCV** - Systematically searches hyperparameters using nested CV

📌 **Cross-Validation** - Provides reliable estimates (mean ± std)

📌 **Regularization** - Controls model complexity (C parameter)

📌 **Split before scale** - ALWAYS split train/test before any transformations

📌 **Test set = vault** - Open ONCE at the end for final evaluation

---

## Production Checklist ✅

**Before deploying any ML model, verify:**

- [ ] Split train/test BEFORE transformations
- [ ] All preprocessing in Pipeline
- [ ] GridSearchCV used for tuning (not manual test set tries)
- [ ] Cross-validation for evaluation
- [ ] Regularization prevents overfitting
- [ ] Test set used ONCE at the end
- [ ] Pipeline saved with joblib
- [ ] No target leakage (features available before prediction)

**If all ✅ → Your model is production-ready!**

---

## Next Steps

1. **Compare with solutions:** `week4_wine_quality_guided_solutions.ipynb`
2. **Take the quiz:** Test your understanding (8 questions, 60% to pass)
3. **Complete post-class exercise:** Build pipeline independently on new dataset (homework)

---

**Great work! You're ready for the quiz!** 🎓

---

*Week 4 Day 2 Guided Practice v1.0 | March 2026*